## 1. Import Libraries

In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np
import lightgbm as lgb
from numerapi import NumerAPI
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
import warnings
warnings.filterwarnings('ignore')

# Add repo root to path
_REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

from utils.metrics import calculate_metrics
from utils.visualization import display_metrics_table
from utils.model_benchmark import record_model_metrics

sns.set_style("whitegrid")
load_dotenv()

True

## 2. Initialization & Data Download

In [2]:
napi = NumerAPI(public_id=os.getenv("NAPI_PUBLIC_ID_UPLOAD"), secret_key=os.getenv("NAPI_SECRET_KEY_UPLOAD"))
DATA_VERSION = "v5.2"
DATA_DIR = "../../data"
os.makedirs(DATA_DIR, exist_ok=True)

# Download feature metadata
napi.download_dataset(f"{DATA_VERSION}/features.json", dest_path=os.path.join(DATA_DIR, DATA_VERSION, "features.json"))
feature_metadata = json.load(open(os.path.join(DATA_DIR, DATA_VERSION, "features.json")))
feature_set = feature_metadata["feature_sets"]["small"]

2026-03-20 15:26:18,143 INFO numerapi.utils: target file already exists
2026-03-20 15:26:18,143 INFO numerapi.utils: download complete


## 3. Data Loading & Pre-processing

In [3]:
print("Loading Data...")

# --- Target Selection via SNNR/Correlation Analysis ---
MAIN_TARGET = "target_ender_20"

snnr_df = pd.read_csv("../../exploratory_notebooks/snnr_weights_vs_correlation_v5.2.csv")
 
AUXILIARY_TARGETS = snnr_df["auxiliary_target"].tolist()
ALL_TARGETS = [MAIN_TARGET] + AUXILIARY_TARGETS

print(f"Main target        : {MAIN_TARGET}")
print(f"Auxiliary targets   ({len(AUXILIARY_TARGETS)}): {AUXILIARY_TARGETS}")
print()

# --- Load Data ---
columns_to_load = ["era"] + feature_set + ALL_TARGETS

train = pd.read_parquet(os.path.join(DATA_DIR, DATA_VERSION, "train.parquet"), columns=columns_to_load)
validation = pd.read_parquet(os.path.join(DATA_DIR, DATA_VERSION, "validation.parquet"),
                             columns=columns_to_load)
live = pd.read_parquet(os.path.join(DATA_DIR, DATA_VERSION, "live.parquet"),
                       columns=["era"] + feature_set)

val_benchmarks = pd.read_parquet(os.path.join(DATA_DIR, DATA_VERSION, "validation_benchmark_models.parquet"))

# Apply 4-era Embargo
last_train_era = int(train["era"].unique()[-1])
eras_to_embargo = [str(era).zfill(4) for era in range(last_train_era + 1, last_train_era + 5)]
validation = validation[~validation["era"].isin(eras_to_embargo)]

print(f"Training data shape   : {train.shape}")
print(f"Validation data shape : {validation.shape}")
print(f"Live data shape       : {live.shape}")


Loading Data...
Main target        : target_ender_20
Auxiliary targets   (17): ['target_jasper_20', 'target_teager2b_20', 'target_claudia_20', 'target_rowan_20', 'target_waldo_20', 'target_ender_60', 'target_xerxes_20', 'target_jeremy_20', 'target_cyrusd_20', 'target_agnes_20', 'target_victor_20', 'target_ralph_20', 'target_caroline_20', 'target_delta_20', 'target_tyler_20', 'target_sam_20', 'target_echo_20']

Training data shape   : (2746268, 61)
Validation data shape : (3921307, 61)
Live data shape       : (7128, 43)


In [4]:
# Reduced data for faster experimentation (last 20 eras, ~10k rows)
# TODO - Turn on or off 
# last_twenty_eras_in_train = train["era"].unique()[-20:]
# print(f"Last 20 eras in training data: {last_twenty_eras_in_train.tolist()}")
# train = train[train["era"].isin(last_twenty_eras_in_train)]
# last_twenty_eras_in_validation = validation["era"].unique()[-20:]
# validation = validation[validation["era"].isin(last_twenty_eras_in_validation)]
# # Print shapes after sampling
# print(f"Sampled training data shape   : {train.shape}")
# print(f"Sampled validation data shape : {validation.shape}")

In [5]:
# Head of training data
print("\nSample of training data:")
display(train.head())
print("\nSample of validation data:")
display(validation.head())


Sample of training data:


,era,feature_antistrophic_striate_conscriptionist,feature_bicameral_showery_wallaba,feature_bridal_fingered_pensioner,feature_collectivist_flaxen_gueux,feature_concurring_fabled_adapter,feature_crosscut_whilom_ataxy,feature_departmental_inimitable_sentencer,feature_dialectal_homely_cambodia,feature_donnard_groutier_twinkle,...,target_jeremy_20,target_cyrusd_20,target_agnes_20,target_victor_20,target_ralph_20,target_caroline_20,target_delta_20,target_tyler_20,target_sam_20,target_echo_20
id,,,,,,,,,,,,,,,,,,,,,
n0007b5abb0c3a25,0001,2,2,2,2,2,0,1,2,2,...,0.25,0.25,0.25,0.25,0.25,0.25,0.00,0.25,0.25,0.00
n003bba8a98662e4,0001,2,2,2,2,2,1,4,2,2,...,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25
n003bee128c2fcfc,0001,2,2,2,2,2,2,2,2,2,...,1.00,0.75,1.00,0.75,0.75,0.75,0.75,1.00,0.75,0.75
n0048ac83aff7194,0001,2,2,2,2,2,1,4,2,2,...,0.50,0.25,0.25,0.50,0.50,0.50,0.50,0.25,0.50,0.50
n0055a2401ba6480,0001,2,2,2,2,2,0,0,2,2,...,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25



Sample of validation data:


,era,feature_antistrophic_striate_conscriptionist,feature_bicameral_showery_wallaba,feature_bridal_fingered_pensioner,feature_collectivist_flaxen_gueux,feature_concurring_fabled_adapter,feature_crosscut_whilom_ataxy,feature_departmental_inimitable_sentencer,feature_dialectal_homely_cambodia,feature_donnard_groutier_twinkle,...,target_jeremy_20,target_cyrusd_20,target_agnes_20,target_victor_20,target_ralph_20,target_caroline_20,target_delta_20,target_tyler_20,target_sam_20,target_echo_20
id,,,,,,,,,,,,,,,,,,,,,
n000c290e4364875,0579,2,2,3,2,2,3,3,2,2,...,0.50,0.50,0.25,0.50,0.50,0.50,0.50,0.5,0.50,0.50
n002a15bc5575bbb,0579,0,4,4,0,1,3,1,1,1,...,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.5,0.25,0.25
n00309caaa0f955e,0579,1,1,4,4,0,3,1,4,3,...,0.50,0.75,0.75,0.50,0.75,0.75,0.75,0.5,0.50,0.75
n0039cbdcf835708,0579,2,2,3,1,4,3,4,3,4,...,0.50,0.50,0.25,0.25,0.50,0.50,0.50,0.5,0.50,0.50
n004143458984f89,0579,4,1,1,1,0,4,4,1,0,...,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.5,0.50,0.50


## 4. Part 1 — Individual Target Model Training & Evaluation

Train one LightGBM model per auxiliary target

In [6]:
def rank_by_era(values, eras):
    """Rank-normalize values within each era to [0, 1]."""
    tmp = pd.DataFrame({
        "era": eras.values,
        "value": np.asarray(values, dtype=float),
    })
    ranked = tmp.groupby("era", group_keys=False)["value"].rank(pct=True, method="first")
    return ranked.values


def neutralize_predictions_by_era(preds, eras, features_df, feature_cols, proportion=0.5, rank_output=True):
    """Neutralize predictions per era against selected features."""
    tmp = pd.DataFrame({
        "era": eras.values,
        "prediction": np.asarray(preds, dtype=float),
    }, index=features_df.index)
    tmp = tmp.join(features_df[feature_cols])

    def _neutralize_block(block):
        scores = block["prediction"].values
        exposures = block[feature_cols].values

        scores = scores - scores.mean()
        exposures = exposures - exposures.mean(axis=0)

        exposure_pinv = np.linalg.pinv(exposures)
        correction = exposures.dot(exposure_pinv.dot(scores))
        neutralized_scores = scores - proportion * correction
        return pd.Series(neutralized_scores, index=block.index)

    neutralized = tmp.groupby("era", group_keys=False).apply(_neutralize_block)
    if rank_output:
        return neutralized.groupby(tmp["era"]).rank(pct=True, method="first").values
    return neutralized.values

In [7]:
from tqdm.auto import tqdm

lgbm_params = {
    "n_estimators": 20000,
    "learning_rate": 0.01,
    "max_depth": 6,
    "num_leaves": 64,
    "colsample_bytree": 0.1,
    "subsample": 0.8,
    "objective": "regression",
    "n_jobs": -1,
    "random_state": 42,
    "verbose": -1,
    "device_type": "gpu",
}

FINAL_NEUTRALIZATION_PROPORTION = 0.5

# Storage for trained models and their per-target results
trained_models = {}                # target_name -> model trained on full training set
oof_predictions = {}               # target_name -> OOF preds on train for meta training
val_predictions_raw = {}           # target_name -> raw predictions on validation
val_predictions_ranked = {}        # target_name -> per-era rank-normalized predictions on validation
live_predictions_raw = {}          # target_name -> raw predictions on live
live_predictions_ranked = {}       # target_name -> per-era rank-normalized predictions on live
member_metrics = []                # list of per-model metric summaries

TARGETS_TO_TRAIN = [t for t in AUXILIARY_TARGETS if t in train.columns and t != MAIN_TARGET]
# TARGETS_TO_TRAIN = TARGETS_TO_TRAIN[:3]  # TODO: turn off for full-target run

print(f">> Final target list for modeling: {TARGETS_TO_TRAIN}")

# Era-based split for stacking: first eras for specialist fit, last eras for OOF/meta training
train_eras_all = np.array(sorted(train["era"].unique()))
stack_eras_count = max(1, int(len(train_eras_all) * 0.2))
base_train_eras = train_eras_all[:-stack_eras_count] if len(train_eras_all) > 1 else train_eras_all
stack_eras = train_eras_all[-stack_eras_count:] if len(train_eras_all) > 1 else train_eras_all

print(f"Base specialist eras : {len(base_train_eras)}")
print(f"Stack/meta eras      : {len(stack_eras)}")

for target_name in tqdm(TARGETS_TO_TRAIN, desc="Training individual specialists"):
    train_clean = train[["era"] + feature_set + [target_name]].dropna()
    if train_clean.empty:
        print(f"Skipping {target_name}: no non-null rows.")
        continue

    base_df = train_clean[train_clean["era"].isin(base_train_eras)]
    stack_df = train_clean[train_clean["era"].isin(stack_eras)]

    if base_df.empty or stack_df.empty:
        print(f"Skipping {target_name}: insufficient data after era split.")
        continue

    specialist = lgb.LGBMRegressor(**lgbm_params)
    specialist.fit(base_df[feature_set], base_df[target_name])

    # OOF-like predictions on holdout stacking eras (used to train meta-model fairly)
    oof_series = pd.Series(np.nan, index=train.index, dtype=float)
    oof_series.loc[stack_df.index] = specialist.predict(stack_df[feature_set])
    oof_predictions[target_name] = oof_series

    # Keep target-level diagnostics on validation using ranked outputs
    val_raw = specialist.predict(validation[feature_set])
    val_ranked = rank_by_era(val_raw, validation["era"])
    val_predictions_raw[target_name] = val_raw
    val_predictions_ranked[target_name] = val_ranked

    live_raw = specialist.predict(live[feature_set])
    live_ranked = rank_by_era(live_raw, live["era"])
    live_predictions_raw[target_name] = live_raw
    live_predictions_ranked[target_name] = live_ranked

    val_eval = validation.copy()
    val_eval["prediction"] = val_ranked

    metrics_dict, per_era_df = calculate_metrics(
        df_validation=val_eval,
        benchmarks=val_benchmarks,
        features=feature_set,
        target_col=MAIN_TARGET,
    )

    snnr_row = snnr_df[snnr_df["auxiliary_target"] == target_name]
    corr_to_main = snnr_row["correlation_with_target"].values[0] if len(snnr_row) else np.nan
    snnr_weight = snnr_row["snnr_weight_normalised"].values[0] if len(snnr_row) else np.nan

    member_metrics.append({
        "target": target_name,
        "corr_to_main": round(float(corr_to_main), 4) if pd.notna(corr_to_main) else np.nan,
        "snnr_weight": round(float(snnr_weight), 4) if pd.notna(snnr_weight) else np.nan,
        "mean_corr": metrics_dict["3_Mean_CORR"],
        "mean_mmc": metrics_dict["4_Mean_MMC"],
        "sharpe_corr": metrics_dict["5_Sharpe_CORR"],
        "sharpe_payout": metrics_dict["2_Sharpe_Payout"],
        "raps": metrics_dict["1_RAPS"],
        "mean_fnc": metrics_dict["6_Mean_FNC"],
        "max_drawdown": metrics_dict["7_Max_Drawdown_CORR"],
        "win_rate": metrics_dict["8_Win_Rate"],
    })

    # Refit specialist on all available train rows for inference (validation/live)
    full_specialist = lgb.LGBMRegressor(**lgbm_params)
    full_specialist.fit(train_clean[feature_set], train_clean[target_name])
    trained_models[target_name] = full_specialist

    val_predictions_raw[target_name] = full_specialist.predict(validation[feature_set])
    val_predictions_ranked[target_name] = rank_by_era(val_predictions_raw[target_name], validation["era"])
    live_predictions_raw[target_name] = full_specialist.predict(live[feature_set])
    live_predictions_ranked[target_name] = rank_by_era(live_predictions_raw[target_name], live["era"])

    print(
        f"  {target_name:30s} | CORR={metrics_dict['3_Mean_CORR']:.5f} "
        f"MMC={metrics_dict['4_Mean_MMC']:.5f} RAPS={metrics_dict['1_RAPS']:.4f}"
    )

print(f"\nTrained {len(trained_models)} individual auxiliary-target specialists.")

>> Final target list for modeling: ['target_jasper_20', 'target_teager2b_20', 'target_claudia_20', 'target_rowan_20', 'target_waldo_20', 'target_ender_60', 'target_xerxes_20', 'target_jeremy_20', 'target_cyrusd_20', 'target_agnes_20', 'target_victor_20', 'target_ralph_20', 'target_caroline_20', 'target_delta_20', 'target_tyler_20', 'target_sam_20', 'target_echo_20']
Base specialist eras : 460
Stack/meta eras      : 114


Training individual specialists:   0%|          | 0/17 [36:26<?, ?it/s]


KeyboardInterrupt: 

### Intermediary Review — Per-Member Performance


In [ ]:
member_df = pd.DataFrame(member_metrics).sort_values("raps", ascending=False).reset_index(drop=True)
member_df.index.name = "rank"

member_df_display = member_df.copy()

round_map = {
    "mean_corr": 5,
    "mean_mmc": 5,
    "sharpe_corr": 3,
    "sharpe_payout": 3,
    "raps": 4,
    "mean_fnc": 5,
    "max_drawdown": 4,
    "corr_to_main": 4,
    "snnr_weight": 4,
}
for col, digits in round_map.items():
    if col in member_df_display.columns:
        member_df_display[col] = member_df_display[col].round(digits)

if "win_rate" in member_df_display.columns:
    member_df_display["win_rate"] = (member_df_display["win_rate"] * 100).round(2).astype(str) + "%"

print("Per-member model metrics (sorted by RAPS)")
display(member_df_display)


----------------------------
# Part 2 The aggregator

In [ ]:
# Build stacking design matrices from specialist predictions
df_train_targets_only = train[["era"] + ALL_TARGETS].dropna(axis=0, how="any")
df_val_targets_only = validation[["era"] + ALL_TARGETS].dropna(axis=0, how="any")

assert MAIN_TARGET in df_train_targets_only.columns, "Main target is missing from training data!"
assert all(target in df_train_targets_only.columns for target in TARGETS_TO_TRAIN), "Some auxiliary targets are missing from training data!"

# Legacy baseline matrices (actual auxiliary targets)
x_train_actual_targets = df_train_targets_only[TARGETS_TO_TRAIN]
y_train_actual_targets = df_train_targets_only[MAIN_TARGET]

x_val_actual_targets = df_val_targets_only[TARGETS_TO_TRAIN]
y_val_actual_targets = df_val_targets_only[MAIN_TARGET]

# OOF stacking matrices for the new method
df_oof_targets = pd.DataFrame(oof_predictions, index=train.index)
df_oof_targets = df_oof_targets.loc[:, TARGETS_TO_TRAIN].dropna(axis=0, how="any")

x_train_predicted_targets = df_oof_targets
y_train_predicted_targets = train.loc[df_oof_targets.index, MAIN_TARGET]

# Validation features built from specialist predictions
df_val_predicted_targets = pd.DataFrame(val_predictions_ranked, index=validation.index)
df_val_predicted_targets = df_val_predicted_targets.loc[df_val_targets_only.index, TARGETS_TO_TRAIN]

x_val_predicted_targets = df_val_predicted_targets
y_val_predicted_targets = y_val_actual_targets.loc[df_val_predicted_targets.index]

assert len(x_train_predicted_targets) == len(y_train_predicted_targets), "Mismatch in OOF stacking train rows."
assert len(x_val_predicted_targets) == len(y_val_predicted_targets), "Mismatch in validation stacking rows."

print("Legacy train matrix shape (actual aux):", x_train_actual_targets.shape)
print("OOF stack train matrix shape (predicted aux):", x_train_predicted_targets.shape)
print("Validation stack matrix shape:", x_val_predicted_targets.shape)

display(x_train_predicted_targets.head())

In [ ]:
# Baseline (old methodology): train meta model on ACTUAL auxiliary targets
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

ridge_old_baseline = Ridge(alpha=1.0e-3, random_state=42)
ridge_old_baseline.fit(x_train_actual_targets, y_train_actual_targets)

ridge_val_predictions_old_actual = ridge_old_baseline.predict(x_val_actual_targets)
ridge_val_predictions_old_on_predicted = ridge_old_baseline.predict(x_val_predicted_targets)

mse_old_actual = mean_squared_error(y_val_actual_targets, ridge_val_predictions_old_actual)
r2_old_actual = r2_score(y_val_actual_targets, ridge_val_predictions_old_actual)

mse_old_on_predicted = mean_squared_error(y_val_actual_targets, ridge_val_predictions_old_on_predicted)
r2_old_on_predicted = r2_score(y_val_actual_targets, ridge_val_predictions_old_on_predicted)

print("Old baseline Ridge (trained on actual aux targets)")
print(f"Validation on actual aux features -> MSE: {mse_old_actual:.6f}, R2: {r2_old_actual:.6f}")
print(f"Validation on predicted aux features -> MSE: {mse_old_on_predicted:.6f}, R2: {r2_old_on_predicted:.6f}")

In [ ]:
# New methodology: train meta model on OOF predicted auxiliary targets
alpha_grid = [1.0e-4, 1.0e-3, 1.0e-2, 1.0e-1, 1.0, 3.0, 10.0, 30.0]
experiments = []

for alpha in alpha_grid:
    for model_name, model in [
        ("Ridge", Ridge(alpha=alpha, random_state=42)),
        ("Lasso", Lasso(alpha=alpha, random_state=42, max_iter=10000)),
    ]:
        model.fit(x_train_predicted_targets, y_train_predicted_targets)

        # Raw and final-neutralized predictions for fair comparison
        preds_raw = model.predict(x_val_predicted_targets)
        preds_ranked = rank_by_era(preds_raw, validation.loc[x_val_predicted_targets.index, "era"])
        preds_final_neutralized = neutralize_predictions_by_era(
            preds=preds_ranked,
            eras=validation.loc[x_val_predicted_targets.index, "era"],
            features_df=validation.loc[x_val_predicted_targets.index],
            feature_cols=feature_set,
            proportion=FINAL_NEUTRALIZATION_PROPORTION,
            rank_output=True,
        )

        mse_raw = mean_squared_error(y_val_predicted_targets, preds_raw)
        r2_raw = r2_score(y_val_predicted_targets, preds_raw)

        eval_df_raw = validation.loc[x_val_predicted_targets.index, ["era"] + feature_set + [MAIN_TARGET]].copy()
        eval_df_raw["prediction"] = preds_ranked
        numerai_raw, _ = calculate_metrics(
            df_validation=eval_df_raw,
            benchmarks=val_benchmarks,
            features=feature_set,
            target_col=MAIN_TARGET,
        )

        eval_df_fn = validation.loc[x_val_predicted_targets.index, ["era"] + feature_set + [MAIN_TARGET]].copy()
        eval_df_fn["prediction"] = preds_final_neutralized
        numerai_fn, _ = calculate_metrics(
            df_validation=eval_df_fn,
            benchmarks=val_benchmarks,
            features=feature_set,
            target_col=MAIN_TARGET,
        )

        coef_series = pd.Series(model.coef_, index=TARGETS_TO_TRAIN)
        coef_df = snnr_df[["auxiliary_target", "snnr_weight_normalised"]].copy()
        coef_df = coef_df[coef_df["auxiliary_target"].isin(TARGETS_TO_TRAIN)]
        coef_df["coef"] = coef_df["auxiliary_target"].map(coef_series.to_dict())
        corr_snnr_coef = coef_df[["snnr_weight_normalised", "coef"]].corr().iloc[0, 1]

        experiments.append({
            "method": "new_oof_stacking",
            "model": model_name,
            "alpha": alpha,
            "mse_raw": mse_raw,
            "r2_raw": r2_raw,
            "mean_corr_raw": numerai_raw["3_Mean_CORR"],
            "raps_raw": numerai_raw["1_RAPS"],
            "mean_corr_final_fn": numerai_fn["3_Mean_CORR"],
            "raps_final_fn": numerai_fn["1_RAPS"],
            "mean_mmc_final_fn": numerai_fn["4_Mean_MMC"],
            "corr_snnr_vs_coef": corr_snnr_coef,
            "nonzero_coef_count": int((np.abs(coef_series.values) > 1e-12).sum()),
            "model_obj": model,
            "val_preds_raw": preds_raw,
            "val_preds_final_fn": preds_final_neutralized,
            "coef_series": coef_series,
        })

experiments_df = pd.DataFrame(experiments).sort_values("raps_final_fn", ascending=False).reset_index(drop=True)

display_cols = [
    "model", "alpha", "mse_raw", "r2_raw", "mean_corr_raw", "raps_raw",
    "mean_corr_final_fn", "raps_final_fn", "mean_mmc_final_fn",
    "corr_snnr_vs_coef", "nonzero_coef_count",
]

print("Meta-learner sweep results (sorted by final FN RAPS):")
display(experiments_df[display_cols].round(6))

best_experiment = experiments_df.iloc[0]
best_model_name = best_experiment["model"]
best_alpha = float(best_experiment["alpha"])
best_meta_model = best_experiment["model_obj"]
best_coef_series = best_experiment["coef_series"]

print(f"Best experiment -> {best_model_name} alpha={best_alpha} with final FN RAPS={best_experiment['raps_final_fn']:.6f}")

In [ ]:
# SNNR vs coefficients diagnostics for the best meta-model
coef_compare_df = snnr_df[["auxiliary_target", "snnr_weight_normalised", "correlation_with_target"]].copy()
coef_compare_df = coef_compare_df[coef_compare_df["auxiliary_target"].isin(TARGETS_TO_TRAIN)]
coef_compare_df["coef"] = coef_compare_df["auxiliary_target"].map(best_coef_series.to_dict())
coef_compare_df["abs_coef"] = coef_compare_df["coef"].abs()
coef_compare_df = coef_compare_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)

corr_matrix = coef_compare_df[["snnr_weight_normalised", "coef", "correlation_with_target"]].corr()

print("Correlation matrix between SNNR weights and meta-model coefficients:")
display(corr_matrix.round(4))

print("Top coefficient contributors (best meta-model):")
display(coef_compare_df.head(20).round(6))

In [ ]:
# Old vs New comparison table (ML + Numerai metrics)
legacy_eval_df = validation.loc[x_val_predicted_targets.index, ["era"] + feature_set + [MAIN_TARGET]].copy()
legacy_eval_df["prediction"] = rank_by_era(ridge_val_predictions_old_on_predicted, legacy_eval_df["era"])
legacy_metrics, _ = calculate_metrics(
    df_validation=legacy_eval_df,
    benchmarks=val_benchmarks,
    features=feature_set,
    target_col=MAIN_TARGET,
 )

best_eval_df = validation.loc[x_val_predicted_targets.index, ["era"] + feature_set + [MAIN_TARGET]].copy()
best_eval_df["prediction"] = best_experiment["val_preds_final_fn"]
best_metrics, _ = calculate_metrics(
    df_validation=best_eval_df,
    benchmarks=val_benchmarks,
    features=feature_set,
    target_col=MAIN_TARGET,
 )

comparison_df = pd.DataFrame([
    {
        "approach": "old_ridge_actual_train_predicted_eval_no_final_fn",
        "model": "Ridge",
        "alpha": 1.0e-3,
        "mse": mse_old_on_predicted,
        "r2": r2_old_on_predicted,
        "mean_corr": legacy_metrics["3_Mean_CORR"],
        "mean_mmc": legacy_metrics["4_Mean_MMC"],
        "raps": legacy_metrics["1_RAPS"],
        "sharpe_payout": legacy_metrics["2_Sharpe_Payout"],
    },
    {
        "approach": "new_oof_stacking_with_final_fn",
        "model": best_model_name,
        "alpha": best_alpha,
        "mse": mean_squared_error(y_val_predicted_targets, best_experiment["val_preds_raw"]),
        "r2": r2_score(y_val_predicted_targets, best_experiment["val_preds_raw"]),
        "mean_corr": best_metrics["3_Mean_CORR"],
        "mean_mmc": best_metrics["4_Mean_MMC"],
        "raps": best_metrics["1_RAPS"],
        "sharpe_payout": best_metrics["2_Sharpe_Payout"],
    },
])

comparison_df["delta_raps_vs_old"] = comparison_df["raps"] - comparison_df.loc[0, "raps"]
comparison_df["delta_corr_vs_old"] = comparison_df["mean_corr"] - comparison_df.loc[0, "mean_corr"]

print("Old vs New methodology comparison:")
display(comparison_df.round(6))

---------------
# Part 3 - Assemble the full ensemble

In [ ]:
# Final ensemble predictions from best meta-model
live_predicted_targets = pd.DataFrame(live_predictions_ranked, index=live.index).loc[:, TARGETS_TO_TRAIN]

ridge_val_predictions_on_preds_final_raw = best_meta_model.predict(x_val_predicted_targets)
ridge_val_predictions_on_preds_final_ranked = rank_by_era(
    ridge_val_predictions_on_preds_final_raw,
    validation.loc[x_val_predicted_targets.index, "era"],
)

ridge_val_predictions_on_preds_final = neutralize_predictions_by_era(
    preds=ridge_val_predictions_on_preds_final_ranked,
    eras=validation.loc[x_val_predicted_targets.index, "era"],
    features_df=validation.loc[x_val_predicted_targets.index],
    feature_cols=feature_set,
    proportion=FINAL_NEUTRALIZATION_PROPORTION,
    rank_output=True,
)

ridge_live_predictions_on_preds_final_raw = best_meta_model.predict(live_predicted_targets)
ridge_live_predictions_on_preds_final_ranked = rank_by_era(
    ridge_live_predictions_on_preds_final_raw,
    live["era"],
)

ridge_live_predictions_on_preds_final = neutralize_predictions_by_era(
    preds=ridge_live_predictions_on_preds_final_ranked,
    eras=live["era"],
    features_df=live,
    feature_cols=feature_set,
    proportion=FINAL_NEUTRALIZATION_PROPORTION,
    rank_output=True,
)

print("Final predictions prepared for validation and live (with single final-stage neutralization).")

In [ ]:
# Evaluate final ensemble predictions with Numerai metrics
eval_df = validation.loc[x_val_predicted_targets.index, ["era"] + feature_set + [MAIN_TARGET]].copy()
eval_df["prediction"] = pd.Series(
    ridge_val_predictions_on_preds_final,
    index=x_val_predicted_targets.index,
    dtype=float,
)

metrics, per_era_df = calculate_metrics(
    df_validation=eval_df,
    benchmarks=val_benchmarks,
    features=feature_set,
    target_col=MAIN_TARGET,
)

display_metrics_table(metrics, extended=True)

In [ ]:
# Compare to benchmark models and record metrics
from pathlib import Path
from utils.model_benchmark import compare_top_models_with_current

model_name = "ridge_meta_ensemble_v0.3.2"
notebook_name = Path(globals().get("__vsc_ipynb_file__", "finance_arena_v0.3.2.ipynb")).stem

history_df, current_run = record_model_metrics(
    metrics=metrics,
    model_name=model_name,
    notebook_name=notebook_name,
 )

leaderboard_df, is_top = compare_top_models_with_current(
    current_run_id=current_run["run_id"],
    top_n=3,
    show_message=True,
 )

print("\nMODEL LEADERBOARD (Top 3 + Current)")
display_df = leaderboard_df.copy()
for col, digits in {"mean_mmc": 6, "mean_corr": 6, "raps": 4, "sharpe_payout": 4}.items():
    if col in display_df.columns:
        display_df[col] = display_df[col].astype(float).round(digits)
display(display_df)